In [ ]:
!pip install --upgrade transformers trl


In [9]:
!pip install latex2sympy2_extended


zsh:1: command not found: pip


In [5]:
# 基础库
import logging
import os
import sys
import re
import math
from dataclasses import dataclass, field
from typing import List, Optional
import warnings

# PyTorch 和 Hugging Face Transformers
import torch
import transformers
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    HfArgumentParser,
    TrainingArguments,
    set_seed,
    TrainerCallback,
    TrainerControl,
    TrainerState,
)
from transformers.trainer_utils import get_last_checkpoint


# 数据集工具
import datasets
from datasets import load_dataset


# TRL（Transformers Reinforcement Learning）库
from trl import (
    AutoModelForCausalLMWithValueHead, 
    PPOConfig, 
    PPOTrainer, 
    GRPOTrainer, 
    GRPOConfig, 
    SFTTrainer,
)

        
# 数学公式处理与验证相关工具
from latex2sympy2_extended import NormalizationConfig
from math_verify import LatexExtractionConfig, parse, verify



In [1]:
!pip show transformers
!pip show trl

zsh:1: command not found: pip
zsh:1: command not found: pip


trl 与 transformers的版本不兼容导致的：

RuntimeError: Failed to import trl.models.modeling_value_head because of the following error (look up to see its traceback):
cannot import name 'HybridCache' from 'transformers' (/opt/anaconda3/envs/gym3.10.10/lib/python3.10/site-packages/transformers/__init__.py)

In [7]:
from transformers.integrations import  is_fsdp_enabled

In [ ]:
# 加载数据集
dataset = load_dataset("../../datasets/NuminaMath-TIR","default")

# 查看数据集结构
print("数据集包含的字段",dataset)
# 获取训练集中的第一个样本
sample = dataset['train'][0]
print(json.dumps(sample,indent=2,ensure_ascii=False))

In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 使用本地模型
local_model_path = "/Users/tao/gittao/models/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(local_model_path)

# 输出 tokenizer 相关信息
print(f"词表大小（Vocabulary size）: {len(tokenizer)}")
print(f"模型最大序列长度（Model max length）: {tokenizer.model_max_length}")
print(f"填充标记（Pad token）: {tokenizer.pad_token}")
print(f"结束标记（EOS token）: {tokenizer.eos_token}")

In [9]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 模型名称与输出目录
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = "data/Qwen-GRPO-training"
local_model_path = "/Users/tao/gittao/models/Qwen2.5-0.5B-Instruct"
# 创建输出目录（如不存在则新建）
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 初始化 tokenizer，使用聊天模板
tokenizer = AutoTokenizer.from_pretrained(
    local_model_path,
    trust_remote_code=True,
    padding_side="right",
)


# 如果没有设置 pad_token，则使用 eos_token 代替
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 输出 tokenizer 相关信息
print(f"词表大小（Vocabulary size）: {len(tokenizer)}")
print(f"模型最大序列长度（Model max length）: {tokenizer.model_max_length}")
print(f"填充标记（Pad token）: {tokenizer.pad_token}")
print(f"结束标记（EOS token）: {tokenizer.eos_token}")

# 加载模型
model = AutoModelForCausalLM.from_pretrained(
    local_model_path,
    trust_remote_code=True,
    dtype=torch.bfloat16  # 使用 bfloat16 更节省显存
)

# 输出模型参数总量
print(f"模型参数总量: {model.num_parameters():,}")

词表大小（Vocabulary size）: 151665
模型最大序列长度（Model max length）: 131072
填充标记（Pad token）: <|endoftext|>
结束标记（EOS token）: <|im_end|>
模型参数总量: 494,032,768


In [ ]:
# 检查设备并加载模型
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"当前使用设备：{device}")
model.to(device)

# 定义基础推理测试函数
def test_model_inference(user_input:str):
    # 使用加载的模型 和 tokenizer进行基础推理测试
    messages = [
        {"role":"system","content":"You are Qwen, a helpful assistant."},
        {"role":"user","content":user_input}
    ]
    # 应用聊天模板，将对话消息转换为模型可以理解的格式
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,# 是否对输出的文本进行分词处理，如True：返回的是分词后的ID列表，False：返回的是格式化后的文本字符串
        add_generation_prompt=True# 指定是否在输出中添加生成提示
    )
    # 编码输入，并移动到设备
    inputs = tokenizer(text,return_tensors="pt").to(device)
    
    # 生成回答
    outs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample= True, # True：模型在生成每个token的时候会从概率分布中随机采样一个令牌，这样会导致生成的文本更加多样化和不可预测 False：模型直接选择概率最高的令牌，生成的文本更加确定和一致
        temperature=0.7
    )
    
    # 解码生成的的结果，取出特殊符号
    response = tokenizer.decode(outs[0],skip_special_tokens=True)
    return response

# 测试模型推理
test_intput = "how are you"
response = test_model_inference(test_intput)
print(f"测试输入：{test_intput}")
print(f"模型回答：{response}")

## 将输入到模型中的数学题数据转换为模型能够理解的对话格式
- 该提示语的核心思想是：将推理过程和最终答案结构化分离，分别包裹在<think>和<answer>标签中，以便后续对不同部分分别进行评估和奖励。

In [ ]:
# DeepSeek GRPO 系统提示
SYSTEM_PROMPT = (
    "A conversation between User and Assistant. The user asks a question, "
    "and the Assistant solves it. The assistant "
    "first thinks about the reasoning process in the mind and "
    "then provides the user with the answer. The reasoning "
    "process and answer are enclosed within <think> </think> "
    "and <answer> </answer> tags, respectively, i.e., "
    "<think> reasoning process here </think><answer> answer here </answer>"
)

### 转换对话格式
- 原始数据中的每道题仅包含题干，我们需要把它包装成一段标准的对话


In [ ]:
# 定义一个函数来处理数据集中的原始样本转换为对话格式
def make_conversation(example):
    """我们将系统提示作为第一轮消息，再将题干作为用户提问，组成一段完整的对话轮次"""
    return {
        "prompt":[
            {"role":"system","content":SYSTEM_PROMPT},
            {"role":"user","content":example['problem']}
        ]
    }

### 加载并预处理数据集


In [ ]:
from datasets import load_dataset

# 加载数学题数据集，并转换为对话格式
def load_math_dataset():
    dataset = load_dataset(
        "/Users/tao/gittao/datasets/NuminaMath-TIR",
        name="default",
        split=['train','test']
    )
    # 拆分成字典形式
    dataset = {
        'train':dataset[0],
        'test':dataset[1]
    }
    # 对训练集 和测试集进行格式转换
    for split  in dataset:
        dataset[split] = dataset[split].map(make_conversation) # 使用map函数应用处理
        # 删除冗余字段
        if 'messages' in dataset[split].column_names:
            dataset[split] = dataset[split].remove_columns("messages")
    return dataset
dataset = load_math_dataset()
print(f"训练集大小:{len(dataset['train'])}")
print(f"测试集大小:{len(dataset['test'])}")

## 奖励函数
- 强化学习的核心驱动力是奖励信号
  - 模型不能真正理解其生成内容的推理是否合理、答案是否正确
  - 它只是依赖奖励反馈不断调整策略，从而逐步逼近人类期望的输出。
  - 因此，奖励函数的设计与实现，不仅定义了“什么是好答案”，更在潜移默化中引导模型”如何思考“
  
- 在DeepSeek R1的训练中，团队未采用传统的大模型评分器，而是构建了一套多维度、可控、具备扩展性的奖励评估体系：（模型会从五个不同的角度对基座模型的回答进行评估）
  - 准确率奖励 ：衡量模型输出在数学意义上是否等价于参考答案。 
  - 格式奖励 ：确保输出结构规范，便于解析和集成。 
  - 推理步骤奖励 ：鼓励展示清晰的推理步骤，提升可解释性。 
  - 余弦缩放奖励 ：在兼顾简洁性与准确性的前提下，对回答长度进行动态调节。 
  - 重复惩罚奖励 ：抑制重复生成，提升输出的多样性。 

### 准确率奖励

准确率奖励是最容易理解的一种方式，目标是判断模型的回答在数学上是否与标准答案等价。
- 如果模型的回答在数学上是正确的，我们会赋予它一个 1.0 的奖励值；
- 如果答案错误，则奖励为 0.0。
- 但如果参考答案由于格式或其他原因无法解析，为了避免不公平的惩罚，我们会给予 0.5 的中性奖励。

In [ ]:
from latex2sympy2_extended import NormalizationConfig
from math_verify import LatexExtractionConfig,parse,verify


"""
模型生成的答案（completions）
相应的标准答案（solution）
"""
def accuracy_reward(completions,solution,**kwargs):
    # 使用latex2sympy2进行Latex表达式解析
    # math_verify 进行等价判断
    # 提取模型生成的内容
    contents = [completion[0]["content"] for completion in completions]
    rewards = []
    
    for content ,sol in zip(contents,solution):
        # 尝试解析标准答案
        gold_parsed = parse(
            sol, # 标准答案
            extraction_mode="first_match",
            extraction_config=[LatexExtractionConfig()]
        )
        if gold_parsed: # 如果解析成功
            # 解析模型答案，使用较为宽松的归一化设置
            answer_parsed = parse(
                content, # 模型解析的答案
                extraction_config=[
                    LatexExtractionConfig(
                        normalization_config=NormalizationConfig(
                            nits=False,
                            malformed_operators=False,
                            basic_latex=True,
                            boxed="all",
                            units=True
                        ),
                        boxed_match_priority=0,
                        try_extract_without_anchor=True,
                    )
                ],
                extraction_mode="first_match",
            )
            reward = float(verify(answer_parsed,gold_parsed))
        else:
            # 如果标准答案解析失败，返回中性奖励0.5，避免错误惩罚
            reward = 0.5
            print("Waring: Failed to parse gold solution:",sol)
        rewards.append(reward)
    return rewards

### 格式奖励（format reward）
格式奖励的核心在于：<b>确保模型按照指令要求输出结构化的回答。</b>
- 我们之前已经要求模型将推理过程放在<think>标签中，最终答案放在<answer>标签中，
- 这个奖励函数的作用就是检查模型是否严格按照我们要求输出内容。

In [ ]:
import  re
def format_reward(completions,**kwargs):
    """
    解释奖励函数：姜茶模型输出是否符合<think>...</think> <answer>...</answer>格式
    
    .*? 是一个非贪婪匹配：
     . 表示匹配任意单个字符（除换行符外）。
     * 表示匹配前面的字符（任意字符）0次或多次。
     ? 表示非贪婪匹配，即尽可能少地匹配字符，直到遇到第一个 </think>
     
     \s*
      \s 表示匹配任意空白字符（包括空格、制表符、换行符等）。
      * 表示匹配前面的空白字符0次或多次。
      这部分的作用是允许在 <think> 和 <answer> 标签之间有任意数量的空白字符（包括没有空白字符）。
    """
    # 正则表达式
    pattern = r"^<think>.*?</think>\s*<answer>.*?</answer>$"
    # 提取模型输出的内容
    completionsn_contents = [completion[0]["content"] for completion in completions]
    # 检查每个输出是否完全符合格式要求
    matches = [ re.match(pattern,content,re.DOTALL | re.MULTILINE) for content in completionsn_contents]
    # 匹配成功 -> 1.0  失败->0.0
    return [1.0 if match else 0.0 for match in matches]

### 推理步骤奖励

推理步骤奖励的目标是鼓励模型展示自己的“思考过程”。我们不希望模型只给出一个答案，而是希望它能够像人类一样，一步步推理，写出解题思路。这个奖励函数的作用，就是给那些包含“推理步骤”的回答打高分。

下面我们来实现这个推理步骤奖励函数，通过识别常见的推理关键词和结构来鼓励模型写过程、讲逻辑。

我们将根据以下特征来判断模型是否展示了清晰的推理过程：
- 明确步骤标记： Step 1 ,  Step 2 ... 
- 有序编号： 1. ,  2. ,  3. 
- 列表符号： - 、 * 
- 逻辑连接词： First ,  Then ,  Next ,  Finally ,  Therefore 等 

上述特征包含得越多，奖励越高。

In [ ]:
import  re

def reasoning_steps_reward(completions,**kwargs):
    """
    鼓励模型输出中包含清晰的思考过程。
    匹配的结构包括step 1、编号、项目符号、逻辑词等
    """
    # 正则表达式：匹配推理步骤线索
    
    pattern = r"(Step \d+:|^\d+\.|\n-|\n\*|First,|Second,|Next,|Finally,)"
    # 提取模型生成的内容
    completion_contents = [completion[0]["contents"] for completion in completions]
    # 匹配每条回答中匹配的推理线索格式
    matches = [len(re.findall(pattern,content,re.MULTILINEUL)) for content in completion_contents]
    # 奖励策略：每出现一个线索加一定分数，3个以上封顶为1.0分
    return [min(1.0,count/3) for count in matches]

### 余弦缩放奖励

余弦缩放奖励的核心理念是：
- 对于正确答案，鼓励模型提供简介清晰的回答
  - 我们更看重回答的简洁性，简短切中要点的回答通常质量更高，会获得更高的奖励
  - 冗长啰嗦的正确答案得分会相对较低
- 对于错误答案，则不会因为回答篇幅较长而给予过度的惩罚
  - 简短错误的回答往往意味着缺乏思考，质量较差
  - 冗长的错误答案，虽然是错误的但是包含一定的推理过程，则更具价值
  - 因为对简短错误回答处罚更严格，对较长错误的回答则相对宽容


In [ ]:
import math
def get_cosin_scaled_reward(
    min_value_wrong:float = -0.5,
    max_value_wrong:float = -0.1,
    min_value_correct:float = 0.8,
    max_value_correct:float = 1.0,
    max_len:int = 1000,
):
    """
    返回一个余弦缩放奖励函数，根据生成答案的长度动态调整准确率奖励
    短的正确答案得分更高
    长的错误答案惩罚较轻
    """
    def cosine_scaled_reward(completions,solution,accuracy_rewards,**kwargs):
        """
        余弦缩放奖励函数，根据答案长度和准确率奖励，计算最终奖励值
        """
        contents = [completion[0]["content"] for completion in completions]
        reward = []
        
       
        for content,acc_reward in zip(contents,accuracy_rewards):
            # 生成答案的长度
            gen_len = len(content)
            # 生成答案长度占最大长度的比例，范围在 [0, 1] 之间。
            progress = gen_len / max_len # 长度占最大长度的比例
            # 余弦值，短答案对应接近1，长答案接近-1
            # 当 progress 接近 0 时，余弦值接近 1；当 progress 接近 1 时，余弦值接近 -1。
            cosine = math.cos(progress * math.pi) # progress * math.pi：将进度值映射到 [0,π]  范围内
            if acc_reward >0.5: # 正确答案
                min_value = min_value_correct
                max_value = max_value_correct
            else: #错误答案，注意反向惩罚
                min_value = max_value_wrong  # 长答案惩罚更轻
                max_value = min_value_wrong  
            # 余弦公式计算
            # 1.0 + cosine 的范围是 [0, 2]
            
            reward = min_value + 0.5 * (max_value - min_value)  * (1.0 + cosine)
            reward.append(float(reward))
        return reward
    return cosine_scaled_reward

In [ ]:
import math
def cosine_scaled_reward(completions,solution,accuracy_rewards,max_len,**kwargs):
        """
        余弦缩放奖励函数，根据答案长度和准确率奖励，计算最终奖励值
        """
        contents = [completion[0]["content"] for completion in completions]
        rewards = []
        min_value_wrong:float = -0.5
        max_value_wrong:float = -0.1
        min_value_correct:float = 0.8
        max_value_correct:float = 1.0
        for content,acc_reward in zip(contents,accuracy_rewards):
            # 生成答案的长度
            gen_len = len(content)
            # 生成答案长度占最大长度的比例，范围在 [0, 1] 之间。
            progress = gen_len / max_len # 长度占最大长度的比例
            # 余弦值，短答案对应接近1，长答案接近-1
            # 当 progress 接近 0 时，余弦值接近 1；当 progress 接近 1 时，余弦值接近 -1。
            cosine = math.cos(progress * math.pi) # progress * math.pi：将进度值映射到 [0,π]  范围内
            if acc_reward >0.5: # 正确答案
                min_value = min_value_correct
                max_value = max_value_correct
            else: #错误答案，注意反向惩罚
                min_value = max_value_wrong  # 长答案惩罚更轻
                max_value = min_value_wrong  
            # 余弦公式计算
            # 1.0 + cosine 的范围是 [0, 2]
            reward = min_value + 0.5 * (max_value - min_value)  * (1.0 + cosine)
            rewards.append(float(reward))
        return rewards

completions = [
    [{"content": "这是一个短答案"}],
    [{"content": "这是一个非常非常非常非常非常非常非常非常长的答案"}]
]
accuracy_rewards = [0.8, 0.3]
rewards = cosine_scaled_reward(completions, "正确答案", accuracy_rewards, max_len=50)
print(rewards)

### 重复惩罚奖励
- 重复惩罚奖励函数的目标是防止模型陷入重复循环，不停地重复相同的句子或短语。
- 我们希望模型能够输出新的、多样化的推理和答案，而不是生成重复内容


- ngram_size 默认取 3，表示统计三元组重复； 
- max_penalty 是最大惩罚分值，必须为负数，如-0.1； 
- 通过计算唯一n-gram占总n-gram比例，得到重复比例 scaling ； 
- 重复比例越高，惩罚越严重，奖励越低（负值）； 
- 空文本或过短文本不惩罚，避免误伤。 

In [ ]:
def get_repetition_penalty_reward(ngram_size:int= 3,max_penalty:float = -0.1):
    """
    返回一个重复惩罚奖励函数，该函数根据生成文本中的n-gram的重复情况给予惩罚
    """
    if max_penalty > 0:
        raise ValueError(f"max_penalty:{max_penalty}")
        
    """
    辅助函数：生成文本中的n-gram
    """
    def zipngram(text:str,ngram_size:int):
        words = text.lower().split()
        return zip(words[i:] for i in range(ngram_size))
    """
    重复惩罚奖励函数实现
    """
    def repetition_penaly_reward(completions,**kwargs)->float:
        contents = [completion[0]['contents'] for completion in completions]
        rewards = []
        for completion in contents:
            if completion == "":
                rewards.append(0.0) # 空回答不惩罚
                continue
            if len(completion.split()) < ngram_size:
                rewards.append(0.0) # 过短文本不惩罚
                continue
            ngrams = set() # 存储唯一的n-gram
            total = 0
            for ng in zipngram(completion,ngram_size):
                ngram.add(ng)
                total+=1
            scaling = 1- len(ngrams) /total # 重复比例越大，重复越多
            reward = scaling * max_penalty # 计算惩罚分数 （负值）
            rewards.append(reward)
        return rewards
    return repetition_penaly_reward

In [ ]:


# 如果 words = ["a", "b", "c", "d"]，ngram_size = 3，则生成的列表为：

[
    ["a", "b", "c", "d"],  # words[0:]
    ["b", "c", "d"],       # words[1:]
    ["c", "d"]             # words[2:]
]

rs = zip(["a", "b", "c", "d"], ["b", "c", "d"], ["c", "d"])
print(list(rs))

# 这会生成一个迭代器，每次迭代返回一个元组，每个元组包含每个子列表的对应元素：
# ("a", "b", "c")
# ("b", "c", "d")

In [ ]:
def zipngram(text:str,ngram_size:int):
        words = text.lower().split()
        return zip(words[i:] for i in range(ngram_size))

text = "I love natural language processing"
ngram_size = 3

ngrams = zipngram(text, ngram_size)
for ngram in ngrams:
    print(ngram)

## 奖励函数配置

In [ ]:
from dataclasses import dataclass,field

"""
GRPO训练脚本中奖励函数配置项
"""
@dataclass
class GRPOScriptArguments:
    reward_funcs :list[str] = field(
        default_factory=lambda:["accuracy","format"],
        metadata={
            'help':"可选择启用的奖励函数：'accuracy', 'format', 'reasoning_steps', 'cosine', 'repetition_penalty'"
        },
    )
    # 余弦相似度奖励相关参数
    cosine_min_value_wrong:float = field(default=-0.5,metadata={"help": "错误答案的最小惩罚值"})
    cosine_max_value_wrong: float = field(default=-0.1, metadata={"help": "错误答案的最大惩罚值"})
    cosine_min_value_correct: float = field(default=0.8, metadata={"help": "正确答案的最小奖励值"})
    cosine_max_value_correct: float = field(default=1.0, metadata={"help": "正确答案的最大奖励值"})
    cosine_max_len: int = field(default=1000, metadata={"help": "用于余弦相似度归一化的最大答案长度"})
    
    # 重复惩罚奖励相关参数
    repetition_n_grams: int = field(default=3, metadata={"help": "重复惩罚中使用的 n-gram 大小"})
    repetition_max_penalty: float = field(default=-0.1, metadata={"help": "重复惩罚的最大负分"})

### 奖励函数动态加载器

In [ ]:
"""
根据脚本参数返回对应的奖励函数列表
"""
def  get_reward_functions(script_args):
    reward_funcs_list = []
    reward_funcs_registry = {
        "accuracy":accuracy_reward, # 准确率奖励函数
        "format":format_reward, # 格式奖励函数
        "reasoning_steps":reasoning_steps_reward, # 推理步骤奖励函数
        "cosine":get_cosin_scaled_reward(
            min_value_wrong=script_args.cosine_min_value_wrong,
            max_value_wrong=script_args.cosine_max_value_wrong,
            min_value_correct=script_args.cosine_min_value_correct,
            max_value_correct=script_args.cosine_max_value_correct,
            max_len=script_args.cosine_max_len,
        ),
        "repetition_penalty":get_repetition_penalty_reward(
            ngram_size=script_args.repetition_n_grams,
            max_penalty=script_args.repetition_max_penalty,
        )
    }
    for func_name in script_args.reward_funcs:
        if func_name not in reward_funcs_registry:
            raise ValueError(f"奖励函数 '{func_name}' %who在注册表中找到。")
        reward_funcs_list.append(reward_funcs_registry[func_name])
    return reward_funcs_list

### 调用这个工具函数，得到奖励函数列表

In [ ]:
script_args = GRPOScriptArguments()
reward_functions = get_reward_functions(script_args)

## R1 训练参数配置

### 训练参数配置

我们使用Hugging Face提供的TrainingArguments类统一管理训练过程的关键参数，比如训练轮次、学习率、保存频率等。

In [14]:
from transformers import TrainingArguments


training_args = TrainingArguments(
    output_dir = OUTPUT_DIR ,#  模型权重和日志保存路径
    overwrite_output_dir = True, # 如果目录存在，是否覆盖
    num_train_epochs = 1, # 训练的轮数
    per_device_train_batch_size = 8, # 训练时每个设备的批大小
    per_device_eval_batch_size = 16, # 验证时每个设备的批大小
    gradient_accumulation_steps = 2, # 梯度累积步数，等下偶增加了批大小
    learning_rate = 55-5, # adamw优化器初始化学习率
    warmup_ratio = 0.1, # 预热比例，线性升温
    weight_decay = 0.01, # 权重衰减率（L2正则）
    logging_steps = 10, # 每多少补打印一次日志
    eval_strategy = "steps", # 以步数为单位进行评估
    eval_steps = 50 ,# 每多少步评估一次
    save_strategy = "steps",# 以步数为单位保存模型
    save_steps = 10, # 每多少步保存一次
    save_total_limit=2, # 最多保存多少个检查点，超出自动删除旧的
    dataloader_num_workers = 2,# 数据加载使用的子进程数
    seed = 42,# 随机种子，保证实验可复现
    # bf16 = True, # 使用BF16混合精度训练，节省显存并提升速度
    push_to_hub = False,# 是否推送模型到huggingface hub
    gradient_checkpointing = True,# 开启梯度检查点，节省显存
    remove_unused_columns = False, # 保留数据集中的所有字段
    report_to = "none", # 关闭训练过程报告（比如不发送到wandb等）
)

### 模型参数配置


ModelConfig类包含以下核心参数：
- 默认使用Qwen2.5-0.5B-Instruct作为基座模型。 
- 启用 bfloat16 提升训练效率。 
- 若硬件支持 flash_attention_2 ，可显著优化注意力计算性能。

In [ ]:

"""
模型配置类
"""

@dataclass
class ModelConfig:
    model_name_or_path : str = field(
        default = MODEL_NAME,metadata={"help":"预训练模型路径或使用huggingface"}
    )
    model_revision: Optional[str] = field(
        default="main",metadata={"help":"指定使用的模型版本，可以是分支名，标签名或提交ID"}
    )
    torch_dtype:Optional[str] = field(
        default="bfloat16",metadata={"help":"覆盖默认的torch_dtype按该数据类型加载模型"}
    )
    trust_remote_code: bool = field(
        default=True,metadata={"help":"加载模型和分词器时是否信息远程代码"}
    )
    attn_implementation:Optional[str] = field(
        default="flash_attention2",metadata={"help":"使用的注意力机制实现支持 'flash_attention_2' 或 None"}
    )

### 训练回调配置

用于在训练过程中定期打印日志，便于实时监控损失与学习率变化。后续可根据需要扩展更多功能型回调。

In [ ]:
import logging
from transformers import TrainerCallback,TrainingArguments,TrainerState,TrainerControl
# 初始化日志记录器
logger = logging.getLogger(__name__)

class LoggingCallback(TrainerCallback):
    # 在每个训练步骤结束时调用
    def on_step_end(self,args:TrainingArguments,state:TrainerState,control:TrainerControl,**kwargs):
        if state.global_step % args.logging_steps ==0:
            if state.log_history:
                logger.info(
                    f"Step{state.global_step}"
                    f"Loss = {state.log_history[-1].get('loss',None)}, "
                    f"Learning Rate = {state.log_history[-1].get('learning_rate',None)}"
                )
            else:
                logger.warning(f"Step {state.global_step}: log_history is still empty.")

                
                
"""
返回训练过程中使用的回调函数列表
当前仅包含LoggingCallback，后续可以根据需要扩展更多功能回调

"""
def get_callbacks(training_args,model_args,script_args):
    callbacks = [LoggingCallback()]
    return callbacks

#### 统一初始化入口

最后，通过调用配置类与回调函数注册函数，完成训练组件的初始化



In [ ]:
script_args = GRPOScriptArguments()
model_args = ModelConfig()

callbacks = get_callbacks(training_args,model_args,script_args)

## GRPO训练与推理

- 在完成模型初始化、数据加载、奖励函数构建、训练参数设置与回调函数配置后，我们即可启动GRPO的训练流程。

- 训练的核心由GRPOTrainer驱动，
  - 它集成了前期准备的各个关键模块：模型、数据集、奖励函数、训练配置与回调机制，负责协调整个训练过程的执行与监控。

In [ ]:
# 从 TrainingArguments 创建 GRPOConfig
grpo_config = GRPOConfig(
    **training_args.to_dict(),  # 将 TrainingArguments 转换为字典并解包传入
    **{
        # 此处省略了 model_init_kwargs
        # 因为我们直接传入了已经初始化好的模型对象，不再需要 GRPOTrainer 内部再做初始化
    }
)
# 构建 GRPOTrainer 实例
grpo_trainer = GRPOTrainer(
    model=model,                      # 已初始化好的 Qwen 模型
    reward_funcs=reward_functions,    # 奖励函数列表
    args=grpo_config,                 # GRPO 配置参数
    train_dataset=dataset['train'],   # 训练数据集
    eval_dataset=dataset['test'],     # 验证数据集
    callbacks=callbacks               # 回调函数列表（如日志、评估等）
)
# 启动 GRPO 训练循环
train_result = grpo_trainer.train()

运行上述代码后，GRPO训练将正式启动。终端将持续输出训练日志，包括损失函数变化、学习率调整等关键信息，便于我们实时监控训练状态与模型收敛情况。
由于本示例中设置num_train_epochs = 1，且模型规模较小，因此训练过程理论上训练时间相对较短，适合快速验证流程。实际训练时长会受硬件条件和数据量等多方面影响，可能仍较耗时。

在真实生产环境中，如DeepSeek R1 Zero的训练任务，通常会采用更多的训练轮次与步数，以保障策略学习的稳定性和模型性能的充分发挥。

## 保存训练后的R1 Zero模型

当训练结束后，我们可以将训练好的模型进行持久化保存，便于后续在推理阶段加载使用，或在此基础上继续进行微调与优化

In [ ]:
# 定义模型保存路径（与训练时设置的 OUTPUT_DIR 保持一致）
TRAINED_MODEL_PATH = "data/Qwen-GRPO-training"
# 保存 tokenizer
tokenizer.save_pretrained(TRAINED_MODEL_PATH)
# 保存训练后的模型权重和配置
grpo_trainer.save_model(TRAINED_MODEL_PATH)
print(f"GRPO 训练完成的模型已保存至：{TRAINED_MODEL_PATH}")

## 加载训练后的r1 zero模型

训练完成后，我们可以加载保存的模型和分词器，用于推理测试或部署上线：

In [ ]:
# 加载tokenizer（如果有远程代码配置，需要启用trust_remote_code=True

tokenizer = AutoTokenizer.from_pretrained(
    TRAINED_MODEL_PATH,
    trust_remote_code=True,
    padding_side = "right" # 确保padding方向一致
)

# 如果pad token未正确保存或加载，手动指定为eos_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 加载训练完成的模型
trained_model = AutoModelForCausalLM.from_pretrained(
    TRAINED_MODEL_PATH,
    trust_remote_code=True,
    torch_dtype = torch_dtype.bfloat16, # 保持与训练阶段一致的数据类型
)
# 将模型移动到目标设备(GPU)
trained_model.to(device)

## 使用训练模型进行推理测试

我们可以编写一个简单的测试函数，调用刚刚加载的模型进行推理：

In [ ]:
# 推理测试函数
def test_trained_model_inference(user_input:str):
    """
        使用已加载模型进行推理测试
    """
    messages = [
        
        {"role":"system","content":SYSTEM_PROMPT},
        {"role":"user","content":user_input}
    ]
    # 应用聊天模板，将消息格式化为模型输入格式
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    
    # 分词并生成张量输入
    inputs = tokenizer(text,return_tensors="pt").to(device)
    # 使用训练好的模型进行生成
    outputs = trained_model.generate(
        **inputs,
        max_new_tokens = 200,# 可适当增加生成长度
        do_sample = True,
        temperature= 0.7, #控制生成多样性
    )
    # 加码生成的结果为可读文本
    response = tokenizer.decode(outputs[0],skip_special_tokens=True)
    return response

In [ ]:
# 我们可以通过调用来测试刚刚训练完成的模型效果，快速验证其响应能力和输出质量
test_trained_model_inference("你好！")

## 构建冷启动数据

为了解决R1 Zero暴露出的推理可读性差、语言混用等问题，并真正提升模型的推理能力，DeepSeek团队在训练初期引入了冷启动数据与监督微调（SFT）的策略。

这一阶段可以理解为：在进行强化学习（RL）训练前，先通过精心标注的高质量数据，教会模型“什么是好的推理过程”，以及“如何清晰、有条理地表达推理逻辑”。这一过程为DeepSeek-V3 Base模型打下推理基础，避免在RL阶段陷入无效探索。

我们此前提到的Bespoke-Stratos-17k数据集，正是典型的冷启动数据集之一。在后续的R1训练中，我们将使用它作为核心的预训练数据源。

当然，要充分发挥冷启动数据的价值，必须深入理解其构建方法和设计理念，确保训练过程中准确传递所有关键信息和策略。

### Few-shot链式思维提示
在冷启动数据的构建过程中，DeepSeek团队采用了一种关键策略：少样本链式思维提示。

通过向模型（如DeepSeek-V3 Base）提供多个示例样本，每个样本都包含推理问题以及其详细的逐步推理过程。这种逐步推理过程，即所谓的Chain-of-Thought（CoT），是提升模型思考能力的重要机制。

通过带有清晰思维路径的示例，模型不仅能学习如何得出正确答案，还逐渐掌握了拆解问题、组织逻辑、分步思考的能力，从而显著提升推理质量与表达结构。

这种训练方法的核心理念是“示范教学”——通过展示人类是如何逐步解决问题，引导模型学习一种清晰、连贯的推理方式。

以下代码展示了如何使用Qwen2.5-0.5B-Instruct模型来生成“逐步推理式”答案

In [ ]:
local_model_path = "/Users/tao/gittao/models/Qwen2.5-0.5B-Instruct"

# 加载模型和分词器
tokenizer = AutoTokenizer.from_pretrained(
    local_model_path,
    trust_remote_code=True,
    padding_side = "right",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    local_model_path,
    trust_remote_code = True,
    dtype = torch.bfloat16
).to("cuda" if torch.cuda.is_available() else "cpu")

# 生成带有推理过程的回答
def generate_response(prompt_text):
    messages = [
        {"role":"system","content":"You are a helpful assistant that provides step-by-step solutions."},
        {"role":"user","content":prompt_text}
        
    ]
    text = tokenizer.apply_chat_template(messages,tokenize=False,add_generation_promt=True)
    inputs = tokenizer(text,return_tensors="pt").to(model.device)
    
    outputs = model.generate(**inputs,max_new_tokens=200,do_sample=False)
    response = tokenizer.decode(outputs[0],skip_special_tokens=True)
    return response.split("<|im_start|>assistant\n")[-1].strip()

我们首先准备几个“示范题目”，每道题都配上了完整的解题步骤和总结，并用一个特殊符号<|special_token|>做分隔标记，方便模型学习格式。

In [ ]:
# 几个带有解题步骤的示例题目
few_shot_prompt = """
Problem: What's the square root of 9 plus 5?
Solution: <|special_token|> First, find the square root of 9, which is 3. Then, add 5 to 3.  3 + 5 equals 8. <|special_token|> Summary: The answer is 8.
Problem: Train travels at 60 mph for 2 hours, how far?
Solution: <|special_token|> Use the formula: Distance = Speed times Time. Speed is 60 mph, Time is 2 hours. Distance = 60 * 2 = 120 miles. <|special_token|> Summary: Train travels 120 miles.
Problem: What is 2 + 3 * 4?
Solution:
"""

接下来我们调用模型，基于以上few-shot示例去解决最后一个问题 “What is 2 + 3 * 4?”：

In [ ]:
# 生成回答内容
model_response_few_shot = generate_response(few_shot_prompt)
# 打印模型的输出
print("Model Response (Few-shot CoT):")
print(model_response_few_shot)

你可以看到，模型在见过前面两个示例后，学会了使用<|special_token|>来分隔解题过程与总结，且能一步步进行逻辑推理。
这就是Few-shot链式思维提示的魅力所在——通过几个高质量示例，就能让模型掌握格式、步骤与推理逻辑，效果甚至接近微调！

### 直接提示

除了使用预先构建的数据集，我们还可以采用一种更加灵活的方式——直接提示（Direct Prompting）。
- 这种方法的核心是：通过精心设计的提示语，引导模型主动生成带有推理过程的回答。

在这种方式下，我们不仅希望模型给出最终答案，更希望它在过程中逐步分析、条理清晰地推理，并对结果进行自我验证。

我们以一道简单的数学题作为例子：
“What is 2 + 3 * 4?”

In [ ]:
# 直接提示
direct_prompt_text = """
Problem: Solve this, show reasoning step-by-step, and verify:
What is 2 + 3 * 4?
"""

model_response_direct = generate_response(direct_prompt_text)
print("Direct Prompt:")
print(direct_prompt_text)
print("\nModel Response (Direct Prompting):")
print(model_response_direct)

通过明确要求“展示推理过程并进行验证”，模型不仅能给出正确答案，还会同步输出完整的思考路径与自我校验过程。这种方式显著提升了模型输出的可解释性与可信度，特别适用于涉及多步推理或复杂思维链条的任务场景。

我们结合Qwen2.5-0.5B-Instruct模型，利用GRPO强化学习策略训练出初始版本的R1 Zero。尽管R1 Zero已具备一定的推理能力，但在推理的清晰度、语言连贯性以及多语言适配性方面仍有不足。

## 冷启动监督微调（Supervised Fine-Tuning, SFT）

可以将冷启动数据收集理解为，在进行高强度的强化学习训练之前，为模型奠定良好的推理基础。
- 其核心目标是，让 DeepSeek-V3 Base 模型（或本文示例中的 Qwen2–0.5B 模型）学习何为高质量的推理，以及如何清晰地呈现推理过程。

我们将详解模型如何借助高质量数据集（如Bespoke-Stratos-17k），逐步学习清晰、有逻辑的推理方式，并通过结构化的输出构建统一、自然的语言风格。

监督微调（SFT）的训练机制：
- 涵盖分词器初始化、
- 训练参数配置、
- 训练主流程及模型保存方法，帮助读者高质量复现微调过程。
此外，我们还将介绍DeepSeek在监督微调第二阶段引入的拒绝采样策略，以及如何通过知识蒸馏将推理能力迁移至更轻量的模型，以支持模型的部署与应用落地。

### 模型训练的内部过程是怎样的？

当我们将数据输入到SFT Trainer后，整体训练流程大致包括以下几个步骤：
1. Tokenization（编码） 将输入输出文本转化为模型可处理的token序列。 
2. Batching（批处理） 将数据按batch分组，提高训练效率，支持并行计算。 
3. Forward Pass（前向传播） 模型根据输入，逐token预测下一个词的概率，生成输出序列。 
4. <b>Loss Calculation（损失计算） 将模型输出与标准答案对比，计算误差（Loss），衡量模型表现。</b> 
5. Backward Pass（反向传播） 基于损失，计算每个参数对误差的梯度（方向和幅度）。 
6. Optimization（参数优化） 使用梯度信息，结合学习率、动量等策略， 更新模型参数 

### R1的监督微调第一阶段（SFT Stage 1）
R1 Zero曾经暴露出的一些问题？

比如：推理过程混乱不清、中英文混杂、语言风格不统一。
- 这些，正是我们希望通过监督微调（SFT）重点修复的内容。

在这个阶段，我们通过引导模型学习：规范、优质的样本，重点提升其推理清晰度与表达一致性，让模型具备结构化、有逻辑、易理解的思考方式，并输出语言统一、自然流畅的回答。

In [17]:
from datasets import load_dataset


bespoke_rl = load_dataset("bespokelabs/Bespoke-Stratos-17k", "default")


# 查看数据集结构
print("数据集包含的字段",bespoke_rl)
# 查看训练集中的第一个样本
bespoke_rl['train'][0]

数据集包含的字段 DatasetDict({
    train: Dataset({
        features: ['system', 'conversations'],
        num_rows: 16710
    })
})


{'system': "Your role as an assistant involves thoroughly exploring questions through a systematic long thinking process before providing the final precise and accurate solutions. This requires engaging in a comprehensive cycle of analysis, summarizing, exploration, reassessment, reflection, backtracing, and iteration to develop well-considered thinking process. Please structure your response into two main sections: Thought and Solution. In the Thought section, detail your reasoning process using the specified format: <|begin_of_thought|> {thought with steps separated with '\\n\\n'} <|end_of_thought|> Each step should include detailed considerations such as analisying questions, summarizing relevant findings, brainstorming new ideas, verifying the accuracy of the current steps, refining any errors, and revisiting previous steps. In the Solution section, based on various attempts, explorations, and reflections from the Thought section, systematically present the final solution that you 

### SFT训练参数配置

使用Hugging Face推出的trl库，它专为强化学习与监督微调任务设计，可以快速构建高质量的训练流程。

- 该配置基本延续了GRPO阶段的优化设置，仅在学习率与batch size上做了适当调整，以更适配监督学习场景。
- 通过启用Flash Attention与梯度检查点，我们能够在不牺牲性能的前提下，显著提升训练效率。

In [17]:
import os
from transformers import TrainingArguments
from trl import ModelConfig

# 模型地址 或 模型名称
local_model_path = "/Users/tao/gittao/models/Qwen2.5-0.5B-Instruct"
# 模型的输出目录，确保与GRPO的阶段的输出路径 不同，避免混淆

SFT_OUTPUT_DIR = "data/Qwen-SFT-training/"

os.makedirs(SFT_OUTPUT_DIR,exist_ok=True)

# 设置SFT截断的训练参数
training_args = TrainingArguments(
    output_dir=SFT_OUTPUT_DIR, # 模型输出路径
    overwrite_output_dir=True, # 如果目录存在则覆盖
    num_train_epochs=1, # 训练轮数，根据需求调整
    per_device_train_batch_size = 8, # 每张卡的训练batch_size
    per_device_eval_batch_size=16, # 每张卡的评估batch_size
    gradient_accumulation_steps=2, # 梯度累积步数，相当于总batch_size扩大2倍
    learning_rate=2e-5, # 学习率适当调低以稳定调试过程
    warmup_ratio=0.1, # 学习率预热比例
    weight_decay=0.01, # 权重衰减，防止过拟合
    logging_steps=10, # 日志打印频率
    eval_steps=50, # 古国开启评估，每50步执行一次
    save_strategy="steps", # 保存策略为按步保存
    save_steps=50, # 每50步保存一次模型
    save_total_limit=2, # 最多保存2个保存点，节省磁盘空间
    dataloader_num_workers=2, # 加载数据的进程数
    seed = 42, # 随机种子，确保实验可复现
    # bf16=True, # 使用bfloat16加速训练
    push_to_hub=False, # 不推送到hub
    gradient_checkpointing=True, # 启用梯度检查点，节省显存
    report_to="none" # 不连接外部日志平台  
)

# 模型配置参数，与GRPO阶段类似，保持高效计算
model_args = ModelConfig(
    model_name_or_path= local_model_path, # 指定模型训练的路径
    model_revision="main", # 模型版本
    dtype="bfloat16", # 模型计算精度
    trust_remote_code=True, # 信任远程代码（信任远程代码）
    # attn_implementation="flash_attention_2" , # 使用Flash Attention v2加速注意力计算
)

### SFT训练主流程

在正式启动监督微调（SFT）之前，我们需要完成两项关键准备工作：加载训练数据集和初始化分词器。

- 我们采用Qwen系列模型配套的分词器，确保分词方式与模型结构完全兼容。
- 训练数据集采用前面提到的高质量指令数据集Bespoke-Stratos-17k：

In [22]:
from datasets import load_dataset
# 加载数据集
dataset_sft = load_dataset("bespokelabs/Bespoke-Stratos-17k", split='train')

from transformers import AutoTokenizer
# 初始化分词
local_model_path = "/Users/tao/gittao/models/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(
    local_model_path,
    trust_remote_code = True,
    padding_side = "right",
)

# 如果模型没有显示设置pad_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

#### 初始化SFTTrainer

In [ ]:
model_sft = AutoModelForCausalLM.from_pretrained(
    local_model_path,
    trust_remote_code = True,
    dtype = torch.bfloat16
)

# 初始化SFT训练器

sft_trainer = SFTTrainer(
    model=model_sft,
    train_dataset=dataset_sft,
    processing_class = tokenizer,
    args = training_args,
)

# 启动监督微调训练循环
sft_train_result = sft_trainer.train()

Converting train dataset to ChatML:   0%|          | 0/16710 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/16710 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/16710 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/opt/anaconda3/envs/gym3.10.10/lib/python3.10/site-packages/pydantic/_internal/_ge

### 保存微调后的tiny r1模型

SFT训练完成后，我们需要将微调后的模型保存下来，便于后续加载和使用。

In [ ]:
# 训练后模型保存路径（与输出目录保持一致）
TRAINED_SFT_MODEL_PATH = "data/Qwen-SFT-training"
# 保存分词器配置
tokenizer.save_pretrained(TRAINED_SFT_MODEL_PATH)
# 保存训练好的模型权重和配置
sft_trainer = save_mode(TRAINED_SFT_MODEL_PATH)
print(f"SFT训练好的模型已经保存至{TRAINED_SFT_MODEL_PATH}")

至此，SFT阶段顺利完成。从基础模型出发，我们通过高质量示例对其进行了系统性的指令微调，让模型学会更清晰、有逻辑地理解问题、组织语言和进行推理。

这个经过第一阶段监督微调的模型，它也将作为后续训练和部署的基础。